In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mabs_funcs as mf
import mfunc_winds as mf2
import matplotlib.colors as colors
import matplotlib.cm as cm

import matplotlib.patches as patches
import itertools


from matplotlib.colors import SymLogNorm
from matplotlib.colors import LogNorm

from matplotlib.ticker import AutoMinorLocator, LogLocator

In [ ]:
plt.rc('text',usetex=True)
from matplotlib import rc
rc('font', **{'family': 'serif', 'serif': ['Computer Modern']})
rc('text', usetex=True)

# Draw triangles

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

def draw_triangle_on_ax(ax, k1, k2, theta,
                        scale=1.0,
                        offset=(0.0, 0.0),
                        rotation=0.0):
    """
    Draw a triangle of sides k1, k2 with included angle theta INTO ax,
    then apply a uniform scale, a rotation about its centroid, and
    a translation (offset) in axes-fraction units.
    
    Parameters
    ----------
    ax : matplotlib.axes.Axes
    k1, k2 : float
        Lengths of the two sides.
    theta : float
        Included angle between k1 and k2 (radians).
    scale : float, optional
        Uniform scale factor (default=1.0).
    offset : tuple of float, optional
        (dx, dy) translation in axes‐fraction coords after transform.
    rotation : float, optional
        Counterclockwise rotation angle in radians about triangle centroid.
    """
    # 1) build the raw triangle in “real” coords
    P0 = np.array([0.0, 0.0])
    P1 = np.array([k1, 0.0])
    P2 = np.array([k2*np.cos(theta), k2*np.sin(theta)])
    pts = np.vstack([P0, P1, P2])

    # 2) normalize to [0,1]×[0,1]
    mn, mx = pts.min(), pts.max()
    norm_pts = (pts - mn) / (mx - mn)

    # 3) compute centroid for rotation
    centroid = norm_pts.mean(axis=0)

    # 4) build rotation matrix
    c, s = np.cos(rotation), np.sin(rotation)
    R = np.array([[c, -s],
                  [s,  c]])

    # 5) apply: move to origin, rotate, move back, then scale & offset
    transformed = (R @ ( (norm_pts - centroid).T )).T + centroid
    transformed = scale * transformed
    transformed = transformed + np.array(offset)

    A, B, C = transformed


#     ax.set_box_aspect(1)

    ax.add_patch(Polygon(transformed, fill=False, edgecolor='black', linewidth=1.5))
    ax.plot([A[0], B[0]], [A[1], B[1]], 'r--', lw=3)
    ax.plot([A[0], C[0]], [A[1], C[1]], 'g--', lw=3)
    ax.plot([B[0], C[0]], [B[1], C[1]], 'b--', lw=3)

    # mids
    midAB = 0.5*(A+B)
    midAC = 0.5*(A+C)
    midBC = 0.5*(B+C)
    ax.text(*midAB+[-.0,-.05], r'$\ell_1$', ha='center', va='top', fontsize=18)
    ax.text(*midAC+[-.0,.02],     r'$\ell_2$', ha='right', va='bottom', rotation=60, fontsize=18)
    ax.text(*midBC+[.02,.02],      r'$\ell_3$', ha='center', va='bottom', rotation=-20, fontsize=18)
    
#     ax.text(*midAB+[-.0,-.05], r'$k_1$', ha='center', va='top', fontsize=18)
#     ax.text(*midAC+[-.0,.02],     r'$k_2$', ha='right', va='bottom', rotation=60, fontsize=18)
#     ax.text(*midBC+[.02,.02],      r'$k_3$', ha='center', va='bottom', rotation=-20, fontsize=18)

    # ensure full view
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xticklabels([]); ax.set_yticklabels([])



In [ ]:
fig, axes = plt.subplots(
    nrows=2, ncols=4,
    gridspec_kw={'height_ratios': [1, 0.4]},
    figsize=(16, 6)
)


# draw_triangle_with_base(ax, x_center=0.5, y_center=0.3, base_len=0.2, side_len=0.13,
#                             box_size=0.33, x_shift=0.0, y_shift=0.0, rotation=0.0):


# Example triangle parameters for the four top-axis panels
params = [
    (1., 1.,   np.pi/3,      .8,   (.12,.15),     0.0),
    (1., 1.,   np.pi*.022421, 1.0, (-.1,.5),      np.pi/4),
    (1., .5,   .1,            1.0, (0,.5),      0.0),
    (1., .772, np.pi*.20049,  1.0, (0,.3),      0.0)
]

for i, (k1, k2, theta, scale, offset, rot) in enumerate(params):
    draw_triangle_on_ax(axes[0, i], k1, k2, theta,scale=scale,offset=offset,rotation=rot)

axes[0,0].text(0.56, 0.112, r'$=r k_{1\perp}$', transform=axes[0,0].transAxes,
        ha='left', va='center', fontsize=18)

tri_names = ['Equilateral', 'Squeezed', 'Stretched', 'Right-angle']
for ind in range(4):
    axes[0,ind].text(0.05, .95, r"%s"%tri_names[ind], transform=axes[0,ind].transAxes, fontsize=25, color="k", ha="left", va="top")
    axes[1,ind].set_xticks([]); axes[1,ind].set_yticks([])
    axes[1,ind].set_xticklabels([]); axes[1,ind].set_yticklabels([])

    ######## second row ##################
ell_values = [
    (710,  710, 710),
    (710,  710, 50),
    (710, 355, 355),
    (710, 548, 419),
]

for ind, (e1, e2, e3) in enumerate(ell_values):
    # build a latex‐math multiline string
    textstr = "\n".join([
        r"$\ell_1 = %d$" % e1,
        r"$\ell_2 = %d$" % e2,
        r"$\ell_3 = %d$" % e3,
    ])
    
    axes[1, ind].text(0.05,    .8,  textstr,transform=axes[1, ind].transAxes,fontsize=17,color='k',ha='left',va='top',linespacing=2)
    
    k1p,k2p,k3p=e1/mf.rc,e2/mf.rc,e3/mf.rc
    textstr_kp = "\n".join([
        r"$k_{1\perp} = %.3f$ Mpc$^{-1}$"%k1p,
        r"$k_{2\perp} = %.3f$ Mpc$^{-1}$"%k2p,
        r"$k_{3\perp} = %.3f$ Mpc$^{-1}$"%k3p,
    ])
    axes[1, ind].text(0.45, .85,  textstr_kp,transform=axes[1, ind].transAxes,fontsize=17,color='k',ha='left',va='top',linespacing=2)
    


fig.tight_layout()
plt.subplots_adjust(left=None, bottom=None, right=None, top=None,wspace=.05, hspace=0)
# plt.show()
# plt.savefig('Plots/tri_conf.pdf', format='pdf', dpi=500, bbox_inches='tight')#, transparent=False), pad_inches=0.1, transparent=False


In [ ]:
k1,k2=710,519
np.sqrt(k1**2 + k2**2 - 2*k1*k2*np.cos(np.pi*.5))

# Shape space plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


# Define the range and resolution for mu and t
mu_values = np.linspace(0.5, 1, 1000)
t_values = np.linspace(0.5, 1, 1000)

# Create a grid of mu and t values
mu, t = np.meshgrid(mu_values, t_values)

k1=1
k2=t*k1
k3=k1*np.sqrt(1-2*mu*t+t*t)

cos_chi=(k3**2 + k2**2 - k1**2)/(2*k2*k3)
cos_chi[999][0]=.6
# Calculate the mu*t values
mu_t = mu * t

# Create a mask to show values only in the specified region
mask = 2 * mu_t >= 1

# Apply the mask to the mu*t values to hide values outside the region
mu_t_masked = np.ma.masked_where(~mask, cos_chi)

# Replace non-zero, unmasked values with 1
mu_t_masked.data[(mu_t_masked.mask == False) & (mu_t_masked.data != 0)] = 1


x=np.arange(.5,1,.001)
x1=np.arange(.707,1,.001)
y=1/(2*x)




gray = (0.827, 0.827, 0.827, 1.0)
custom_cmap = mcolors.ListedColormap([gray])




# Create a color plot
plt.figure(figsize=(8, 6))
plt.rc('text',usetex=True)
from matplotlib import rc
rc('font', **{'family': 'serif', 'serif': ['Computer Modern']})
rc('text', usetex=True)
plt.plot(x,y,'-',linewidth=2,color='k')
plt.plot(x1,x1,'--',linewidth=1.5,color='black')
plt.imshow(mu_t_masked, cmap=custom_cmap, extent=[0.5, 1, 0.5, 1], origin='lower', aspect='equal')

plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('$\mu$',fontsize=28)
plt.ylabel('$t$',fontsize=28)

plt.text(0.39, .99, 'Equilateral', fontsize=20, rotation=45)
plt.text(0.98, 0.98, 'Squeezed', fontsize=20, rotation=-45)
plt.text(.985, 0.425, 'Stretched', fontsize=20, rotation=45)


plt.text(.71, 1.013, 'L Isosceles', fontsize=20, rotation=0)
plt.text(1.01, 0.7, 'Linear', fontsize=20, rotation=90)
plt.text(.68, 0.91, 'Acute', fontsize=22, rotation=0)
plt.text(.85, 0.66, 'Obtuse', fontsize=22, rotation=0)
plt.text(.74, 0.775, 'Right angle', fontsize=20, rotation=45)
plt.text(.64, 0.63, 'S Isosceles', fontsize=20, rotation=-47)

# arrow_props = dict(arrowstyle="<|-|>", lw=1.5)
# plt.annotate("", xy=(1,1 ), xytext=(1.02, .5), arrowprops=arrow_props, fontsize=12, ha='center', va='center')

# plt.arrow(0.15, 0.55, 1, 0, head_width=0.05, head_length=0.03, linewidth=4, color='r', length_includes_head=True)
plt.annotate('', xy=(1.04, 1), xycoords='axes fraction', xytext=(1.04, .55),arrowprops=dict(arrowstyle="->", color='black',lw=1.3))
plt.annotate('', xy=(1.04, .37), xycoords='axes fraction', xytext=(1.04, 0),arrowprops=dict(arrowstyle="<-", color='black',lw=1.3))


plt.annotate('', xy=(.68, 1.04), xycoords='axes fraction', xytext=(1, 1.04),arrowprops=dict(arrowstyle="<-", color='black',lw=1.15))
plt.annotate('', xy=(.41, 1.04), xycoords='axes fraction', xytext=(0, 1.04),arrowprops=dict(arrowstyle="<-", color='black',lw=1.15))

x=np.arange(.748,.9,.001)
y=1/(2*x)-.038
plt.plot(x,y,'-',linewidth=1.3,color='black')
plt.annotate('', xy=(.7, .1), xycoords='axes fraction', xytext=(.82, .022),arrowprops=dict(arrowstyle="<-", color='black',lw=1.))


x=np.arange(.515,.649,.001)
y=1/(2*x)-.038
plt.plot(x,y,'-',linewidth=1.3,color='black')
plt.annotate('', xy=(.05, .828), xycoords='axes fraction', xytext=(0.012, .9),arrowprops=dict(arrowstyle="<-", color='black',lw=1.1))


plt.hlines([.999],0.5,1,linewidth=2,color='k')
plt.vlines([.999],0.5,1,linewidth=2,color='k')

for ii in [0.6,0.7,0.8,0.9]:
    plt.hlines([ii],1/(2*ii),1,ls=':',linewidth=.9,color='brown')
    plt.vlines([ii],1/(2*ii),1,ls=':',linewidth=.9,color='brown')



# fig.tight_layout()
# plt.subplots_adjust(left=0, bottom=None, right=1.1, top=None,wspace=0.25, hspace=0)
# plt.savefig('Plots/shape_space.png', format='png', dpi=500, bbox_inches='tight')  

In [ ]:
mu_t_masked

# extra

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.colors as mcolors


# # Define the range and resolution for mu and t
# mu_values = np.linspace(0.5, 1, 1000)
# t_values = np.linspace(0.5, 1, 1000)

# # Create a grid of mu and t values
# mu, t = np.meshgrid(mu_values, t_values)

# k1=1
# k2=t*k1
# k3=k1*np.sqrt(1-2*mu*t+t*t)

# cos_chi=(k3**2 + k2**2 - k1**2)/(2*k2*k3)
# cos_chi[999][0]=.6
# # Calculate the mu*t values
# mu_t = mu * t

# # Create a mask to show values only in the specified region
# mask = 2 * mu_t >= 1

# # Apply the mask to the mu*t values to hide values outside the region
# mu_t_masked = np.ma.masked_where(~mask, cos_chi)


# x=np.arange(.5,1,.001)
# x1=np.arange(.707,1,.001)
# y=1/(2*x)





# # Define the colormap and the number of discrete colors
# cmap = plt.get_cmap('Spectral') #RdYlGn_r
# num_colors = 16

# # Create a list of equally spaced values in the mu*t range
# values = np.linspace(0, 1, num_colors)

# # Create a color map with only the desired number of colors
# norm = plt.Normalize(0, 1)
# colors = [cmap(norm(value)) for value in values]

# # Create a custom colormap with the selected colors
# custom_cmap = mcolors.ListedColormap(colors)


# ################ Start figure ###################

# fig, axes = plt.subplots(1,2,figsize=(12,6))

# plt.rc('text',usetex=True)
# from matplotlib import rc
# rc('font', **{'family': 'serif', 'serif': ['Computer Modern']})
# rc('text', usetex=True)


# ########### axes[0]#################
# ax=axes[0]
# ax.set_xticks([]); ax.set_yticks([])
# ax.set_xticklabels([]); ax.set_yticklabels([])
# draw_triangle_on_ax(ax, k1=1., k2=.72, theta=np.pi*.20049,scale=.95,offset=(0.01,.48),rotation=0)
# ax.text(0.14, 0.513, r'$\theta$', transform=ax.transAxes,ha='left', va='center', fontsize=18)
# ax.text(0.53, 0.8, r'$\alpha$', transform=ax.transAxes,ha='left', va='center', fontsize=18)


# ax.text(0.2, 0.25, r'$\mu=\cos(\theta)$', transform=ax.transAxes,ha='left', va='center', fontsize=22)
# ax.text(0.2, 0.1, r'$t=\frac{k_2}{k_1}$', transform=ax.transAxes,ha='left', va='center', fontsize=22)

# ax.set_box_aspect(1)

# ############ axes[1] #########################
# ax=axes[1]

# ax.plot(x, y, '--', linewidth=1.5, color='black')
# ax.plot(x1, x1, '--', linewidth=1.5, color='black')
# im = ax.imshow(mu_t_masked, cmap=custom_cmap, extent=[0.5, 1, 0.5, 1], origin='lower', aspect='equal')


# # Set ticks and labels using ax
# ax.tick_params(labelsize=15)
# ax.set_xlabel('$\mu$', fontsize=20)  # Unicode for mu
# ax.set_ylabel('$t$', fontsize=20)  # Unicode for t (tau, or use 't' if preferred)

# ax.text(0.41, .995, 'Equilateral', fontsize=19, rotation=37)
# ax.text(0.98, 0.98, 'Squeezed', fontsize=19, rotation=-45)
# ax.text(.985, 0.425, 'Stretched', fontsize=19, rotation=45)
# ax.text(.71, 1.013, 'L Isosceles', fontsize=20, rotation=0)
# ax.text(1.01, 0.7, 'Linear', fontsize=20, rotation=90)
# ax.text(.68, 0.89, 'Acute', fontsize=22, rotation=0)
# ax.text(.85, 0.66, 'Obtuse', fontsize=22, rotation=0)
# ax.text(.74, 0.775, 'Right angle', fontsize=20, rotation=45)
# ax.text(.64, 0.63, 'S Isosceles', fontsize=20, rotation=-47)

# # arrow_props = dict(arrowstyle="<|-|>", lw=1.5)
# # plt.annotate("", xy=(1,1 ), xytext=(1.02, .5), arrowprops=arrow_props, fontsize=12, ha='center', va='center')

# # plt.arrow(0.15, 0.55, 1, 0, head_width=0.05, head_length=0.03, linewidth=4, color='r', length_includes_head=True)
# ax.annotate('', xy=(1.04, 1), xycoords='axes fraction', xytext=(1.04, .55),arrowprops=dict(arrowstyle="->", color='black',lw=1.3))
# ax.annotate('', xy=(1.04, .37), xycoords='axes fraction', xytext=(1.04, 0),arrowprops=dict(arrowstyle="<-", color='black',lw=1.3))

# ax.annotate('', xy=(.68, 1.04), xycoords='axes fraction', xytext=(1, 1.04),arrowprops=dict(arrowstyle="<-", color='black',lw=1.15))
# ax.annotate('', xy=(.41, 1.04), xycoords='axes fraction', xytext=(0, 1.04),arrowprops=dict(arrowstyle="<-", color='black',lw=1.15))

# x=np.arange(.748,.9,.001)
# y=1/(2*x)-.038
# ax.plot(x,y,'-',linewidth=1.3,color='black')
# ax.annotate('', xy=(.7, .1), xycoords='axes fraction', xytext=(.82, .022),arrowprops=dict(arrowstyle="<-", color='black',lw=1.))


# x=np.arange(.515,.649,.001)
# y=1/(2*x)-.038
# ax.plot(x,y,'-',linewidth=1.3,color='black')
# ax.annotate('', xy=(.05, .828), xycoords='axes fraction', xytext=(0.012, .9),arrowprops=dict(arrowstyle="<-", color='black',lw=1.1))





# cax = fig.add_axes([.965, .145, .02, 0.7]) #fig.add_axes([.55, -.05, .36, 0.02]) # [left, bottom, width, height]
# cbar = fig.colorbar(im, cax=cax, ticks=np.arange(-1, 0.7, .2), orientation='vertical',location='right')
# cbar.ax.set_ylabel('cos$(\\alpha)$', fontsize=20, labelpad=15)
# cbar.ax.tick_params(labelsize=15)


# # fig.tight_layout()
# # plt.subplots_adjust(left=0, bottom=None, right=1.1, top=None,wspace=0.25, hspace=0)
# # plt.savefig('Plots/shape_space.pdf', format='pdf', dpi=500, bbox_inches='tight')  

In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches
# import numpy as np

# def draw_triangle_with_base(ax, x_center=0.5, y_center=0.3, base_len=0.2, side_len=0.13,
#                             box_size=0.33, x_shift=0.0, y_shift=0.0, rotation=0.0):
#     """
#     Draw an isosceles triangle (two equal sides, base user-defined) in a fixed square box,
#     with optional position shift and rotation.

#     Parameters:
#     - ax: matplotlib axis
#     - x_center, y_center: center of the fixed square box (axes coords)
#     - base_len: length of triangle's base
#     - side_len: length of the two equal sides
#     - box_size: size of the square box (axes fraction)
#     - x_shift, y_shift: shift triangle inside the square (axes fraction)
#     - rotation: rotation angle in degrees (clockwise) around triangle center
#     """
#     # Compute triangle center position
#     tri_x = x_center + x_shift
#     tri_y = y_center + y_shift

#     # Half base
#     half_base = base_len / 2

#     # Triangle vertices before rotation
#     A = np.array([-half_base, 0])  # left base
#     B = np.array([half_base, 0])   # right base
#     height = np.sqrt(side_len**2 - half_base**2)
#     C = np.array([0, height])      # apex

#     triangle = np.array([A, B, C])

#     # Apply rotation
#     theta = np.deg2rad(rotation)
#     rotation_matrix = np.array([[np.cos(theta), -np.sin(theta)],
#                                  [np.sin(theta),  np.cos(theta)]])
#     triangle_rotated = triangle @ rotation_matrix.T

#     # Shift triangle to its position in axes coords
#     triangle_rotated += np.array([tri_x, tri_y])

#     # Plot triangle
#     colors = ['orange', 'red', 'blue']
#     for i in range(3):
#         x0, y0 = triangle_rotated[i]
#         x1, y1 = triangle_rotated[(i + 1) % 3]
#         ax.plot([x0, x1], [y0, y1], color='k', ls='-', transform=ax.transAxes,linewidth=1)
#         ax.plot([x0, x1], [y0, y1], color=colors[i], ls='--', transform=ax.transAxes,linewidth=1.5)

#     # Add side labels at midpoints
# #     for i in range(3):
# #         mid = (triangle_rotated[i] + triangle_rotated[(i + 1) % 3]) / 2
# #         ax.text(mid[0], mid[1], r'$\ell$', fontsize=14, ha='center', va='center', transform=ax.transAxes)

#     # Draw fixed square box
# #     box_x = x_center - box_size / 2
# #     box_y = y_center - box_size / 2
# #     box = patches.Rectangle((box_x, box_y), box_size, box_size,
# #                             linewidth=1, edgecolor='black', facecolor='none', transform=ax.transAxes)
# #     ax.add_patch(box)


In [ ]:
fig, ax = plt.subplots(
    nrows=2, ncols=4,
    gridspec_kw={'height_ratios': [1, 0.5]},
    figsize=(12, 6)
)

draw_triangle_with_base(ax[0,0], x_center=0.5, y_center=0.3, base_len=.7, side_len=.5,
                            box_size=0.33, x_shift=0.0, y_shift=0.0, rotation=0.0)